# Bee migration tutorial

A walk through the `bee_migration` package: build observations, look at trends
and correlations, then train models that forecast bee abundance.

The observations here are **simulated** from assumed warming and decline rates.
They show how the method works; they are not field measurements.

Install the package first: `pip install -e ".[notebook]"`.

## 1. Observations

In [ ]:
from bee_migration.climate.period import DEFAULT_PERIOD
from bee_migration.climate.region import BRAZILIAN_REGIONS
from bee_migration.climate.simulation import ClimateSimulation

simulation = ClimateSimulation(BRAZILIAN_REGIONS, DEFAULT_PERIOD)
observations = simulation.observations(seed=42)

observations.table().head()

Each row is one region in one year. Besides the measurements, the table carries
indices derived from them: anomalies against the regional mean, a climate stress
index and a bee health index.

To use your own data, load a CSV with the same measurement columns:

```python
from pathlib import Path
from bee_migration.climate.observations import Observations

observations = Observations.from_csv(Path("my_observations.csv"))
```

## 2. Trends

In [ ]:
from bee_migration.analysis.temporal import TemporalPatterns

patterns = TemporalPatterns(observations)
patterns.trends().round(3)

Values are the change per year. Years where a series bends sharply:

In [ ]:
patterns.change_points()

## 3. Correlations

In [ ]:
from bee_migration.analysis.correlation import Correlations

correlations = Correlations(observations)
[pair.describe() for pair in correlations.notable_pairs()[:5]]

Temperature against bee abundance one to three years later, by region:

In [ ]:
correlations.lagged().round(2)

## 4. Regions

In [ ]:
from bee_migration.analysis.geographic import GeographicPatterns

GeographicPatterns(observations).hotspots().round(2)

## 5. Forecasting

Four models learn bee abundance from climate and from the previous year's value.
The four most recent years are held out: no model sees them while training.
`QUICK_CANDIDATES` uses fixed hyperparameters; `DEFAULT_CANDIDATES` searches for
them and takes longer.

In [ ]:
from bee_migration.prediction.catalog import QUICK_CANDIDATES
from bee_migration.prediction.training import MigrationPredictor

leaderboard = MigrationPredictor(QUICK_CANDIDATES, holdout_years=4).train(observations)
leaderboard.metrics()[["test_r2", "test_rmse", "train_r2"]].round(3)

What the best model relies on, measured by shuffling each feature:

In [ ]:
best = leaderboard.best()
print(best.name)
best.importance.sort_values(ascending=False).head(8).round(3)

## 6. Charts

The chart functions return matplotlib figures, which Jupyter displays when they
are the last value of a cell.

In [ ]:
from bee_migration.charts import observed
from bee_migration.climate.continental import continental_series

series = continental_series(DEFAULT_PERIOD, seed=42).set_index("year")
observed.temperature_migration_trend(series)

## Next steps

- `bee-migration all` runs everything above from the command line and writes the
  reports, tables and charts.
- The [dashboard](https://digomes87.github.io/Bees/) runs the same package in the
  browser and adds live field data from GBIF and Open-Meteo.